In [1]:
print(123)

123


In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\ITPB-RH\.conda\envs\llm-zoomcamp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1287.55it/s]


In [5]:
v1 = model.encode(q1)

In [6]:
v1.shape

(384,)

In [7]:
v2 = model.encode(q2)

In [8]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [9]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [10]:
v1.dot(dv)

np.float32(0.3233239)

In [11]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [12]:
v2.dot(dv)

np.float32(0.019730505)

In [13]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py


"wget" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [29]:
import sys
sys.path.append('../util/')
from ingest import load_faq_data

documents = load_faq_data()

In [61]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [62]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [63]:
len(texts)

1350

In [64]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [34]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

100%|██████████| 27/27 [01:58<00:00,  4.38s/it]


1350

In [70]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [71]:
import numpy as np
X = np.array(vectors)

In [68]:
X[:1]

array([[-2.67062075e-02, -1.22457527e-01,  1.59441940e-02,
         6.09419867e-03, -1.11915218e-02, -6.55021369e-02,
        -7.62883723e-02, -2.08819825e-02, -9.27567780e-02,
         3.55664939e-02,  3.15623283e-02, -1.09017240e-02,
        -2.45336164e-02, -1.82476491e-02,  3.43917459e-02,
        -1.32052237e-02,  7.22679775e-03, -1.54126689e-01,
         6.41842633e-02, -9.07447562e-03,  3.94655354e-02,
        -2.99638007e-02,  2.08512526e-02,  3.71391885e-02,
         3.52526829e-02, -2.49496219e-03,  7.71129057e-02,
         2.78048366e-02,  1.54832313e-02,  4.92820097e-03,
         1.48517417e-03,  3.93927172e-02,  7.25188628e-02,
         9.02841091e-02,  5.25653400e-02,  2.72272509e-02,
         3.86085436e-02, -7.50263184e-02, -2.48754360e-02,
         1.06018238e-01, -4.82870899e-02, -5.00598289e-02,
        -4.16486450e-02,  7.91618004e-02,  5.06461747e-02,
        -3.68654437e-04, -2.83320178e-03, -2.59598251e-02,
        -3.82818282e-02,  8.59545320e-02, -2.85154935e-0

In [73]:
scores = X.dot(v1)

In [74]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629409))

In [75]:
documents[553]

{'course': 'llm-zoomcamp',
 'section': 'Module 1: RAG',
 'question': 'OpenAI: Error when running OpenAI responses.create command',
 'answer': 'You may receive the following error when running the OpenAI `responses.create` command due to insufficient credits in your OpenAI account:\n\n```\nOpenAI API Error: Insufficient credits\n```',
 'doc_id': 'f5df151c59'}

In [76]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [77]:
scores[top5]

array([0.7629409 , 0.757937  , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [78]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629409
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.757937
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [43]:
top5

array([  2, 625, 907, 538,   7])

In [44]:
top5 = np.argsort(-scores)[:5]

In [45]:
top5

array([  2, 625, 907, 538,   7])

In [46]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [47]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [66]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


--2026-05-18 10:56:54--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-05-18 10:56:54 (25.3 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [53]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [54]:
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPENAI_TOKEN")
endpoint = os.getenv("ENDPOINT_URL")
deployment_name = os.getenv("DEPLOYMENT_NAME")

In [55]:

openai_client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

In [49]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [56]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [57]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)


'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [58]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [59]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [60]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)


'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [76]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [77]:
vs_index.fit(vectors, documents)


In [80]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [81]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [86]:
vs_index.close()